# Ejercicio Stream
#### Gabriel García 21352

## Parte 1: Implementación del Stream Cipher (70 puntos)
### 1.1 Generación del Keystream (20 puntos)
### 1.2 Función de Cifrado (25 puntos)
### 1.3 Función de Descifrado (25 puntos)

In [10]:
import random
import hashlib

In [ ]:
# Convertimos la clave a bytes de forma consistente
def _key_to_seed_int(key):
    if isinstance(key, bytes):
        key_bytes = key
    else:
        key_bytes = str(key).encode("utf-8")

    # Hacemos un hash para obtener un seed estable (misma clave => mismo seed)
    digest = hashlib.sha256(key_bytes).digest()

    # Convertimos el hash (bytes) a un entero
    return int.from_bytes(digest, byteorder="big")
    
# PRNG básico: Mersenne Twister (random.Random) inicializado con seed
def generar_keystream(clave, longitud):
    seed_int = _key_to_seed_int(clave)
    prng = random.Random(seed_int)

    # Generamos 'longitud' bytes pseudoaleatorios (0..255)
    ks = bytearray()
    i = 0
    while i < longitud:
        ks.append(prng.randrange(256))
        i += 1

    return bytes(ks)

def cifrar_stream(mensaje, clave):
    # Acepta mensaje como str o bytes
    if isinstance(mensaje, bytes):
        mensaje_bytes = mensaje
    else:
        mensaje_bytes = str(mensaje).encode("utf-8")

    # Keystream del mismo tamaño del mensaje
    ks = generar_keystream(clave, len(mensaje_bytes))

    # XOR byte a byte
    cifrado = bytearray()
    i = 0
    while i < len(mensaje_bytes):
        cifrado.append(mensaje_bytes[i] ^ ks[i])
        i += 1

    return bytes(cifrado)

def descifrar_stream(ciphertext, clave):
    # En stream cipher, descifrar es lo mismo: XOR con el mismo keystream
    if not isinstance(ciphertext, bytes):
        raise TypeError("ciphertext debe ser bytes")

    # Generamos el mismo keystream con la misma clave y la misma longitud
    ks = generar_keystream(clave, len(ciphertext))

    # XOR byte a byte para recuperar el texto plano en bytes
    plano = bytearray()
    i = 0
    while i < len(ciphertext):
        plano.append(ciphertext[i] ^ ks[i])
        i += 1

    return bytes(plano)

In [12]:

mensaje = "Hola mundo"
clave = "mi_clave"

longitud_necesaria = len(mensaje.encode("utf-8"))
ks1 = generar_keystream(clave, longitud_necesaria)
ks2 = generar_keystream(clave, longitud_necesaria)

print("Longitud mensaje en bytes:", longitud_necesaria)
print("Keystream 1 (hex):", ks1.hex())
print("Keystream 2 (hex):", ks2.hex())
print("Deterministico (ks1 == ks2):", ks1 == ks2)


Longitud mensaje en bytes: 10
Keystream 1 (hex): fb2b6a962afaab4e7523
Keystream 2 (hex): fb2b6a962afaab4e7523
Deterministico (ks1 == ks2): True
